# 05 - Campaign Efficiency (LeakLens)
**Goal:** Measure true campaign efficiency after removing impossible rows and accounting for returns where possible.

**Approach:** pure, transparent calculation (no ML model needed). The saved object holds the scoring logic.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

DATA = Path("../data")
MODELS = Path("../models")
MODELS.mkdir(exist_ok=True)

def first_existing(df, names):
    for n in names:
        if n in df.columns:
            return n
    return None

def to_bool(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "yes", "y", "t"])

In [2]:
campaigns = pd.read_csv(DATA / "campaigns.csv")
transactions = pd.read_csv(DATA / "transactions.csv")
products = pd.read_csv(DATA / "products.csv")

print("campaigns    :", campaigns.shape)
print("transactions :", transactions.shape)
print("products     :", products.shape)

campaigns    : (1200, 8)
transactions : (18000, 10)
products     : (250, 5)


In [3]:
campaigns["impossible"] = False
if "conversions" in campaigns.columns and "clicks" in campaigns.columns:
    campaigns["impossible"] = campaigns["conversions"] > campaigns["clicks"]

print("Impossible campaigns (conversions > clicks):", int(campaigns["impossible"].sum()),
      "of", len(campaigns))

Impossible campaigns (conversions > clicks): 88 of 1200


In [4]:
tx = transactions.copy()

# Bring category onto transactions if needed
if "category" not in tx.columns and "category" in products.columns and "product_id" in tx.columns:
    tx = tx.merge(products[["product_id", "category"]].drop_duplicates("product_id"),
                  on="product_id", how="left")

tx_date = first_existing(tx, ["transaction_date", "date", "order_date", "timestamp"])
camp_start = first_existing(campaigns, ["start_date", "date", "campaign_date"])
camp_end = first_existing(campaigns, ["end_date"])

# Sales amount column (or quantity * price)
amount_col = first_existing(tx, ["net_amount", "total_amount", "amount", "revenue", "sales"])
if amount_col is None and "quantity" in tx.columns:
    price_col = first_existing(tx, ["unit_price", "price"])
    if price_col:
        tx["_amount"] = tx["quantity"] * tx[price_col]
        amount_col = "_amount"

# Returns: boolean flag or a returned-amount column
return_flag_col = first_existing(tx, ["is_returned", "returned", "return_flag"])
return_amt_col = first_existing(tx, ["return_amount", "refund_amount"])

can_link = all([tx_date, camp_start, amount_col, "channel" in tx.columns,
                "channel" in campaigns.columns, "category" in tx.columns,
                "category" in campaigns.columns])
print("Can link transactions to campaigns:", bool(can_link))
print("Sales column:", amount_col, "| Return column:", return_flag_col or return_amt_col)

campaigns["linked_sales"] = np.nan
campaigns["linked_returns"] = np.nan

if can_link:
    tx[tx_date] = pd.to_datetime(tx[tx_date], errors="coerce")
    campaigns[camp_start] = pd.to_datetime(campaigns[camp_start], errors="coerce")
    if camp_end:
        campaigns[camp_end] = pd.to_datetime(campaigns[camp_end], errors="coerce")

    if return_flag_col:
        tx["_returned"] = np.where(to_bool(tx[return_flag_col]), tx[amount_col], 0.0)
    elif return_amt_col:
        tx["_returned"] = tx[return_amt_col].fillna(0)
    else:
        tx["_returned"] = 0.0

    sales_list, returns_list = [], []
    for _, row in campaigns.iterrows():
        start = row[camp_start]
        end = row[camp_end] if camp_end else start
        mask = ((tx["channel"] == row["channel"]) &
                (tx["category"] == row["category"]) &
                (tx[tx_date] >= start) & (tx[tx_date] <= end))
        sales_list.append(tx.loc[mask, amount_col].sum())
        returns_list.append(tx.loc[mask, "_returned"].sum())

    campaigns["linked_sales"] = sales_list
    campaigns["linked_returns"] = returns_list
    print("Campaigns with linked sales > 0:", int((campaigns["linked_sales"] > 0).sum()))

Can link transactions to campaigns: True
Sales column: _amount | Return column: None
Campaigns with linked sales > 0: 1


In [5]:
def per_spend(num, spend):
    return np.where(spend > 0, num / np.where(spend > 0, spend, 1), np.nan)

spend = campaigns["spend"] if "spend" in campaigns.columns else pd.Series(np.nan, index=campaigns.index)

if "conversions" in campaigns.columns:
    campaigns["conv_per_spend"] = per_spend(campaigns["conversions"], spend)
campaigns["sales_per_spend"] = per_spend(campaigns["linked_sales"].fillna(0), spend)
campaigns["net_sales"] = campaigns["linked_sales"] - campaigns["linked_returns"]
campaigns["net_sales_per_spend"] = per_spend(campaigns["net_sales"].fillna(0), spend)

# Choose the primary metric: return-adjusted sales if linking worked, else conversions
if can_link and (campaigns["linked_sales"] > 0).any():
    primary_metric = "net_sales_per_spend"
else:
    primary_metric = "conv_per_spend"
print("Primary efficiency metric:", primary_metric)

# Impossible rows get no efficiency score (their numbers cannot be trusted)
campaigns.loc[campaigns["impossible"], primary_metric] = np.nan

clean_vals = campaigns[primary_metric].dropna()
reference = np.sort(clean_vals.values)
campaigns["efficiency_score"] = campaigns[primary_metric].apply(
    lambda v: np.nan if pd.isna(v) else 100 * np.searchsorted(reference, v, side="right") / len(reference)
)
print("Scored campaigns:", len(clean_vals), "| Excluded (impossible/no spend):", len(campaigns) - len(clean_vals))

Primary efficiency metric: net_sales_per_spend
Scored campaigns: 1112 | Excluded (impossible/no spend): 88


In [6]:
id_cols = [c for c in ["campaign_id", "channel", "category"] if c in campaigns.columns]
metric_cols = [c for c in ["spend", "conversions", "linked_sales", "linked_returns",
                           "conv_per_spend", "sales_per_spend", "net_sales_per_spend"] if c in campaigns.columns]

ranked = (campaigns[~campaigns["impossible"]]
          .dropna(subset=[primary_metric])
          .sort_values(primary_metric, ascending=False)
          .reset_index(drop=True))
ranked.index = ranked.index + 1
ranked.index.name = "rank"

show = id_cols + metric_cols + ["efficiency_score"]
print("TOP 10 most efficient campaigns:")
display(ranked[show].head(10).round(3))
print("BOTTOM 10 least efficient campaigns:")
display(ranked[show].tail(10).round(3))

print("Impossible campaigns excluded from ranking:", int(campaigns["impossible"].sum()))

if "channel" in campaigns.columns:
    print("\nAverage efficiency by channel:")
    print(ranked.groupby("channel")[primary_metric].mean().sort_values(ascending=False).round(4))

TOP 10 most efficient campaigns:


,campaign_id,channel,category,spend,conversions,linked_sales,linked_returns,conv_per_spend,sales_per_spend,net_sales_per_spend,efficiency_score
rank,,,,,,,,,,,
1,CA0071,App,Electronics,7903.17,1174,239.16,0.0,0.149,0.03,0.03,100.00
2,CA0001,Email,Electronics,3467.47,1047,0.00,0.0,0.302,0.00,0.00,99.91
3,CA0795,Social,Sports,5091.47,588,0.00,0.0,0.115,0.00,0.00,99.91
4,CA0801,Store,Electronics,3886.34,541,0.00,0.0,0.139,0.00,0.00,99.91
5,CA0800,Store,Fashion,1213.49,673,0.00,0.0,0.555,0.00,0.00,99.91
6,CA0799,Email,Beauty,4342.93,1027,0.00,0.0,0.236,0.00,0.00,99.91
7,CA0798,App,Electronics,4199.77,198,0.00,0.0,0.047,0.00,0.00,99.91
8,CA0797,Store,Sports,3178.20,1202,0.00,0.0,0.378,0.00,0.00,99.91
9,CA0796,Email,Grocery,7602.68,1267,0.00,0.0,0.167,0.00,0.00,99.91


BOTTOM 10 least efficient campaigns:


,campaign_id,channel,category,spend,conversions,linked_sales,linked_returns,conv_per_spend,sales_per_spend,net_sales_per_spend,efficiency_score
rank,,,,,,,,,,,
1103,CA0402,Social,Fashion,6057.67,1273,0.0,0.0,0.210,0.0,0.0,99.91
1104,CA0403,Store,Home,5593.69,415,0.0,0.0,0.074,0.0,0.0,99.91
1105,CA0404,Web,Home,7561.12,1262,0.0,0.0,0.167,0.0,0.0,99.91
1106,CA0405,Web,Grocery,7989.61,501,0.0,0.0,0.063,0.0,0.0,99.91
1107,CA0406,Web,Electronics,3802.30,879,0.0,0.0,0.231,0.0,0.0,99.91
1108,CA0407,Web,Grocery,3813.61,1014,0.0,0.0,0.266,0.0,0.0,99.91
1109,CA0408,Web,Sports,6202.88,163,0.0,0.0,0.026,0.0,0.0,99.91
1110,CA0409,Web,Grocery,7928.93,22,0.0,0.0,0.003,0.0,0.0,99.91
1111,CA0410,Social,Home,1912.94,1382,0.0,0.0,0.722,0.0,0.0,99.91


Impossible campaigns excluded from ranking: 88

Average efficiency by channel:
channel
App       0.0001
Email     0.0000
Social    0.0000
Store     0.0000
Web       0.0000
Name: net_sales_per_spend, dtype: float64


In [7]:
artifact = {
    "primary_metric": primary_metric,
    "reference_distribution": reference,   # used to turn a metric into a 0-100 score
    "impossible_rule": "conversions > clicks",
    "link_keys": ["date range", "channel", "category"],
    "sales_column": amount_col,
    "return_column": return_flag_col or return_amt_col,
}
joblib.dump(artifact, MODELS / "campaign_model.joblib")
print("Saved:", MODELS / "campaign_model.joblib")

Saved: ..\models\campaign_model.joblib


## Summary for judges
- **Trustworthy numbers first:** campaigns with impossible data (conversions > clicks) are removed before ranking, so bad rows cannot look like top performers.
- **True efficiency:** campaigns are ranked by return-adjusted sales per unit of spend when transactions can be linked by date, channel and category. Otherwise they fall back to conversions per spend.
- **No black box:** this is pure calculation, with a 0-100 percentile score anyone can verify.
- **Business value:** the bottom of the table shows where money leaks (high spend, low net return), and the top shows where to move budget.